# Modelaje del ETA en tres fases operativas

Este notebook rehace el flujo de trabajo dividiendo el ETA total en tres subproblemas de predicción:

1. **Fase 1 — Activación → inicio del courier**  
   Tiempo hasta que el courier empieza el pedido.
2. **Fase 2 — Inicio del courier → pickup**  
   Tiempo de desplazamiento/espera hasta la recogida.
3. **Fase 3 — Pickup → entrega**  
   Última milla hasta el cliente.

La idea no es únicamente maximizar el score, sino construir transformaciones justificables desde el punto de vista operacional.

## 0. Carga de librerías y datos

El notebook asume que el archivo `glovo_ops_data_final.csv` está en el entorno de Colab.  
Si no lo encuentra, abrirá un selector para subirlo manualmente.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [ ]:
DATA_PATH = Path("glovo_ops_data_final.csv")

if not DATA_PATH.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = Path(next(iter(uploaded.keys())))
    except Exception:
        # Ruta útil si se ejecuta localmente en el entorno de esta conversación.
        DATA_PATH = Path("/mnt/data/glovo_ops_data_final.csv")

df_raw = pd.read_csv(DATA_PATH)
print(df_raw.shape)
df_raw.head()

## 1. Inspección inicial

Antes de transformar, revisamos tipos, nulos y cardinalidad.  
Esto permite decidir qué columnas conviene usar como predictores y cuáles tienen riesgo de fuga de información.

In [ ]:
display(df_raw.info())

summary = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_%": df_raw.isna().mean().mul(100).round(2),
    "n_unique": df_raw.nunique()
}).sort_values(["missing_%", "n_unique"], ascending=False)

summary

## 2. Definición de las tres variables objetivo

Se crean tres targets en minutos:

- `phase_1_assignment_minutes`: desde `activation_time_local` hasta `courier_started_order_local`.
- `phase_2_pickup_minutes`: desde `courier_started_order_local` hasta `pickup_time_local`.
- `phase_3_last_mile_minutes`: desde `pickup_time_local` hasta `delivery_time_local`.

También se conserva el ETA total (`delivery_time_minutes`) para comprobar que las fases tienen sentido.

### Nota importante sobre la fase 1

En los datos hay casos donde `courier_started_order_local` ocurre antes de `activation_time_local`.  
Eso puede indicar preasignación, sincronización imperfecta de sistemas o una definición operacional distinta de “activación”.

Para modelar un tiempo físico no negativo, usamos:

```python
phase_1_assignment_minutes = max(0, courier_started_order_local - activation_time_local)
```

y guardamos el indicador `phase_1_preassigned_flag` para el análisis, pero **no lo usamos como predictor del modelo**, porque se deriva del target.

In [ ]:
df = df_raw.copy()

# Corrección de typo en la columna original
df = df.rename(columns={"tranport_type": "transport_type"})

time_cols = [
    "activation_time_local",
    "courier_started_order_local",
    "pickup_time_local",
    "delivery_time_local"
]

for col in time_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Targets por fase
df["phase_1_raw_minutes"] = (
    df["courier_started_order_local"] - df["activation_time_local"]
).dt.total_seconds() / 60

df["phase_1_preassigned_flag"] = (df["phase_1_raw_minutes"] < 0).astype(int)
df["phase_1_assignment_minutes"] = df["phase_1_raw_minutes"].clip(lower=0)

df["phase_2_pickup_minutes"] = (
    df["pickup_time_local"] - df["courier_started_order_local"]
).dt.total_seconds() / 60

df["phase_3_last_mile_minutes"] = (
    df["delivery_time_local"] - df["pickup_time_local"]
).dt.total_seconds() / 60

df["delivery_time_minutes"] = (
    df["delivery_time_local"] - df["activation_time_local"]
).dt.total_seconds() / 60

phase_targets = [
    "phase_1_assignment_minutes",
    "phase_2_pickup_minutes",
    "phase_3_last_mile_minutes",
    "delivery_time_minutes"
]

phase_summary = (
    df[phase_targets + ["phase_1_raw_minutes"]]
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
    .T
)

phase_summary["negative_values"] = [
    (df["phase_1_assignment_minutes"] < 0).sum(),
    (df["phase_2_pickup_minutes"] < 0).sum(),
    (df["phase_3_last_mile_minutes"] < 0).sum(),
    (df["delivery_time_minutes"] < 0).sum(),
    (df["phase_1_raw_minutes"] < 0).sum()
]

phase_summary

In [ ]:
# Comprobación de descomposición: ETA total ≈ suma de fases
df["sum_operational_phases"] = (
    df["phase_1_assignment_minutes"]
    + df["phase_2_pickup_minutes"]
    + df["phase_3_last_mile_minutes"]
)

df[[
    "delivery_time_minutes",
    "sum_operational_phases",
    "phase_1_assignment_minutes",
    "phase_2_pickup_minutes",
    "phase_3_last_mile_minutes"
]].head()

## 3. Feature engineering

Se crean variables que tienen lógica operacional:

### Variables espaciales
- Distancia Haversine entre pickup y entrega.
- Diferencia absoluta de latitud y longitud.

### Variables temporales
- Hora y día codificados de forma cíclica con seno/coseno.
- Fin de semana.
- Hora pico aproximada.

### Variables del pedido
- Número aproximado de productos a partir de `description`.
- Longitud de la descripción.
- Valor del pedido (`gtv`).
- Delivery fee.

### Variables históricas sin target leakage
- Número de pedidos previos del cliente.
- Número de pedidos previos de la tienda.
- Número de pedidos previos del courier, solo usable cuando el courier ya está asignado.

No se usan `rating` ni `bad_rating_reason` como predictores porque son información posterior a la entrega.

In [ ]:
def haversine_np(lat1, lon1, lat2, lon2):
    """Distancia Haversine en km entre dos puntos."""
    R = 6371.0
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 2 * R * np.arcsin(np.sqrt(a))


def extract_num_items(text):
    """Extrae cantidades del patrón 'N x producto'. Si no aparece, asume 1 item."""
    if pd.isna(text):
        return 0

    text = str(text).lower()
    nums = re.findall(r"(\d+)\s*x", text)

    if nums:
        # Capping para evitar valores absurdos por parsing
        return min(sum(map(int, nums)), 100)

    return 1


def add_cyclical_time_features(data, timestamp_col, prefix):
    """Añade hora y día de semana con codificación cíclica."""
    data[f"{prefix}_hour"] = data[timestamp_col].dt.hour
    data[f"{prefix}_dow"] = data[timestamp_col].dt.dayofweek

    data[f"{prefix}_hour_sin"] = np.sin(2 * np.pi * data[f"{prefix}_hour"] / 24)
    data[f"{prefix}_hour_cos"] = np.cos(2 * np.pi * data[f"{prefix}_hour"] / 24)

    data[f"{prefix}_dow_sin"] = np.sin(2 * np.pi * data[f"{prefix}_dow"] / 7)
    data[f"{prefix}_dow_cos"] = np.cos(2 * np.pi * data[f"{prefix}_dow"] / 7)

    return data


def limit_cardinality(series, top_n=40, other_label="Other"):
    """Agrupa categorías poco frecuentes para evitar one-hot encoding excesivo."""
    series = series.fillna("Unknown").astype(str)
    top_values = series.value_counts().head(top_n).index
    return np.where(series.isin(top_values), series, other_label)


df = df.sort_values("activation_time_local").reset_index(drop=True)

# Features espaciales
df["distance_km"] = haversine_np(
    df["pickup_latitude"],
    df["pickup_longitude"],
    df["delivery_latitude"],
    df["delivery_longitude"]
)

df["lat_diff_abs"] = (df["delivery_latitude"] - df["pickup_latitude"]).abs()
df["lon_diff_abs"] = (df["delivery_longitude"] - df["pickup_longitude"]).abs()

# Features del pedido
df["num_items"] = df["description"].apply(extract_num_items)
df["description_length"] = df["description"].fillna("").astype(str).str.len()

# Features temporales
df = add_cyclical_time_features(df, "activation_time_local", "activation")
df = add_cyclical_time_features(df, "courier_started_order_local", "start")
df = add_cyclical_time_features(df, "pickup_time_local", "pickup")

df["is_weekend"] = df["activation_time_local"].dt.dayofweek.isin([5, 6]).astype(int)

# Picos aproximados: comida y cena
df["is_lunch_peak"] = df["activation_time_local"].dt.hour.between(12, 14).astype(int)
df["is_dinner_peak"] = df["activation_time_local"].dt.hour.between(19, 22).astype(int)

# Festivos franceses. Si la librería no está disponible, se deja la feature en 0.
try:
    import holidays
except ImportError:
    try:
        get_ipython().system("pip -q install holidays")
        import holidays
    except Exception:
        holidays = None

if holidays is not None:
    years = sorted(df["activation_time_local"].dt.year.dropna().unique().astype(int))
    fr_holidays = holidays.country_holidays("FR", years=years)
    df["is_holiday"] = df["activation_time_local"].dt.date.astype("object").isin(fr_holidays).astype(int)
else:
    df["is_holiday"] = 0

# Features históricas sin usar el target
df["customer_past_orders"] = df.groupby("customer_id", dropna=False).cumcount()
df["store_past_orders"] = df.groupby("store_address_id", dropna=False).cumcount()
df["courier_past_orders"] = df.groupby("courier_id", dropna=False).cumcount()

# Categóricas limpias
df["vertical_clean"] = df["vertical"].fillna("Unknown").astype(str)
df["store_name_clean"] = limit_cardinality(df["store_name"], top_n=40)
df["transport_type"] = df["transport_type"].fillna("Unknown").astype(str)

df[[
    "distance_km",
    "num_items",
    "description_length",
    "is_weekend",
    "is_lunch_peak",
    "is_dinner_peak",
    "is_holiday",
    "customer_past_orders",
    "store_past_orders",
    "courier_past_orders",
    "vertical_clean",
    "store_name_clean",
    "transport_type"
]].head()

## 4. EDA específico de las fases

Ahora el análisis se centra en la distribución de cada subtiempo, no solo del ETA total.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

plot_targets = [
    ("phase_1_assignment_minutes", "Fase 1: activación → inicio courier"),
    ("phase_2_pickup_minutes", "Fase 2: inicio courier → pickup"),
    ("phase_3_last_mile_minutes", "Fase 3: pickup → entrega")
]

for ax, (target, title) in zip(axes, plot_targets):
    clean_values = df[target].dropna()
    clean_values = clean_values[(clean_values >= 0) & (clean_values <= clean_values.quantile(0.995))]
    ax.hist(clean_values, bins=50)
    ax.set_title(title)
    ax.set_xlabel("minutos")
    ax.set_ylabel("frecuencia")

plt.tight_layout()
plt.show()

In [ ]:
phase_by_transport = (
    df
    .query("phase_2_pickup_minutes >= 0 and phase_3_last_mile_minutes >= 0")
    .groupby("transport_type")[[
        "phase_1_assignment_minutes",
        "phase_2_pickup_minutes",
        "phase_3_last_mile_minutes",
        "delivery_time_minutes"
    ]]
    .agg(["count", "mean", "median"])
)

phase_by_transport

In [ ]:
phase_by_vertical = (
    df
    .query("phase_2_pickup_minutes >= 0 and phase_3_last_mile_minutes >= 0")
    .groupby("vertical_clean")[[
        "phase_1_assignment_minutes",
        "phase_2_pickup_minutes",
        "phase_3_last_mile_minutes",
        "delivery_time_minutes"
    ]]
    .agg(["count", "mean", "median"])
)

phase_by_vertical

## 5. Limpieza por fase

No todas las fases necesitan la misma limpieza:

- Fase 1: se modela con clipping inferior a 0, porque el tiempo de espera no puede ser negativo.
- Fase 2 y fase 3: los valores negativos se eliminan, porque indican orden temporal imposible.
- En todas las fases se elimina la cola extrema superior con el percentil 99.5.

Esta limpieza es más defendible que aplicar un único filtro global al ETA total.

In [ ]:
def clean_phase_data(data, target_col, upper_q=0.995):
    """Limpieza específica para cada fase."""
    clean = data.dropna(subset=[target_col, "activation_time_local"]).copy()

    # En fase 2 y fase 3 los negativos son inconsistencias claras.
    # En fase 1 ya se hizo clipping a 0.
    clean = clean[clean[target_col] >= 0]

    upper = clean[target_col].quantile(upper_q)
    clean = clean[clean[target_col] <= upper]

    return clean


cleaning_report = []

for target in [
    "phase_1_assignment_minutes",
    "phase_2_pickup_minutes",
    "phase_3_last_mile_minutes"
]:
    before = len(df)
    after = len(clean_phase_data(df, target))
    cleaning_report.append({
        "target": target,
        "rows_before": before,
        "rows_after": after,
        "rows_removed": before - after,
        "removed_%": round((before - after) / before * 100, 2)
    })

pd.DataFrame(cleaning_report)

## 6. Separación de variables por disponibilidad operacional

La clave del modelaje por fases es no usar información futura.

### Fase 1 — predicción al activarse el pedido
Disponibles:
- tienda/vertical,
- valor y fee,
- descripción,
- pickup/delivery coordinates,
- saturación,
- hora y día de activación.

No se usa:
- courier,
- pickup time,
- delivery time,
- rating.

### Fase 2 — predicción cuando el courier empieza el pedido
Disponibles:
- todo lo anterior,
- tipo de transporte,
- duración real de fase 1,
- hora de inicio del courier,
- historial de pedidos del courier.

### Fase 3 — predicción al hacer pickup
Disponibles:
- todo lo anterior,
- duración real de fase 2,
- hora de pickup.

Este planteamiento permite construir un ETA dinámico que se actualiza a medida que el pedido avanza.

In [ ]:
base_numeric_features = [
    "gtv",
    "delivery_fee",
    "saturation",
    "distance_km",
    "lat_diff_abs",
    "lon_diff_abs",
    "num_items",
    "description_length",
    "activation_hour_sin",
    "activation_hour_cos",
    "activation_dow_sin",
    "activation_dow_cos",
    "is_weekend",
    "is_lunch_peak",
    "is_dinner_peak",
    "is_holiday",
    "customer_past_orders",
    "store_past_orders"
]

base_categorical_features = [
    "vertical_clean",
    "store_name_clean"
]

phase_config = {
    "Fase 1 - asignación/inicio": {
        "target": "phase_1_assignment_minutes",
        "numeric": base_numeric_features,
        "categorical": base_categorical_features
    },

    "Fase 2 - llegada a pickup": {
        "target": "phase_2_pickup_minutes",
        "numeric": base_numeric_features + [
            "phase_1_assignment_minutes",
            "start_hour_sin",
            "start_hour_cos",
            "start_dow_sin",
            "start_dow_cos",
            "courier_past_orders"
        ],
        "categorical": base_categorical_features + [
            "transport_type"
        ]
    },

    "Fase 3 - última milla": {
        "target": "phase_3_last_mile_minutes",
        "numeric": base_numeric_features + [
            "phase_1_assignment_minutes",
            "phase_2_pickup_minutes",
            "pickup_hour_sin",
            "pickup_hour_cos",
            "pickup_dow_sin",
            "pickup_dow_cos",
            "courier_past_orders"
        ],
        "categorical": base_categorical_features + [
            "transport_type"
        ]
    }
}

for phase, cfg in phase_config.items():
    print("\n", phase)
    print("Target:", cfg["target"])
    print("Num features:", len(cfg["numeric"]))
    print("Cat features:", len(cfg["categorical"]))

## 7. Preprocesamiento y modelos

Se comparan tres modelos por fase:

1. `DummyRegressor`: baseline que predice la media.
2. `Ridge`: modelo lineal regularizado.
3. `RandomForestRegressor`: modelo no lineal que captura interacciones.

El split es **temporal**, no aleatorio: el 80% inicial se usa para entrenar y el 20% final para test.  
Esto se parece más a una situación real, donde se entrena con el pasado y se predice sobre pedidos futuros.

In [ ]:
def make_ohe():
    """Compatibilidad entre versiones de scikit-learn."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocessor(numeric_features, categorical_features):
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_ohe())
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features)
        ],
        sparse_threshold=0
    )


def temporal_split(data, time_col="activation_time_local", test_size=0.20):
    data = data.sort_values(time_col).reset_index(drop=True)
    split_idx = int(len(data) * (1 - test_size))

    train = data.iloc[:split_idx].copy()
    test = data.iloc[split_idx:].copy()

    return train, test


def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred)
    }


def get_models():
    return {
        "DummyMean": DummyRegressor(strategy="mean"),
        "Ridge": Ridge(alpha=1.0),
        "RandomForest": RandomForestRegressor(
            n_estimators=200,
            min_samples_leaf=20,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    }


def train_and_evaluate_phase(data, phase_name, config):
    target = config["target"]
    numeric_features = config["numeric"]
    categorical_features = config["categorical"]

    phase_df = clean_phase_data(data, target_col=target)
    train_df, test_df = temporal_split(phase_df)

    X_train = train_df[numeric_features + categorical_features]
    y_train = train_df[target]

    X_test = test_df[numeric_features + categorical_features]
    y_test = test_df[target]

    results = []
    fitted_models = {}

    for model_name, model in get_models().items():
        pipe = Pipeline(steps=[
            ("preprocess", make_preprocessor(numeric_features, categorical_features)),
            ("model", model)
        ])

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        row = {
            "phase": phase_name,
            "target": target,
            "model": model_name,
            "n_train": len(train_df),
            "n_test": len(test_df),
            **regression_metrics(y_test, y_pred)
        }

        results.append(row)
        fitted_models[(phase_name, model_name)] = pipe

    return pd.DataFrame(results), fitted_models

In [ ]:
all_results = []
all_models = {}

for phase_name, config in phase_config.items():
    print(f"Entrenando: {phase_name}")
    phase_results, phase_models = train_and_evaluate_phase(df, phase_name, config)
    all_results.append(phase_results)
    all_models.update(phase_models)

results_df = pd.concat(all_results, ignore_index=True)
results_df.sort_values(["phase", "MAE"])

## 8. Interpretación de importancia de variables

Para el modelo Random Forest, revisamos qué variables pesan más por fase.

La interpretación esperada sería:

- Fase 1: saturación, hora, tienda/vertical.
- Fase 2: tienda, vertical, número de items, saturación, fase 1.
- Fase 3: distancia, transporte, saturación y hora.

In [ ]:
def get_feature_names_from_pipeline(pipe):
    preprocess = pipe.named_steps["preprocess"]

    try:
        return preprocess.get_feature_names_out()
    except Exception:
        # Fallback manual
        names = []
        for name, transformer, cols in preprocess.transformers_:
            if name == "num":
                names.extend(cols)
            elif name == "cat":
                ohe = transformer.named_steps["onehot"]
                try:
                    cat_names = ohe.get_feature_names_out(cols)
                except Exception:
                    cat_names = ohe.get_feature_names(cols)
                names.extend(cat_names)
        return np.array(names)


def plot_rf_importance(models_dict, phase_name, top_n=20):
    key = (phase_name, "RandomForest")
    pipe = models_dict[key]

    feature_names = get_feature_names_from_pipeline(pipe)
    importances = pipe.named_steps["model"].feature_importances_

    imp_df = (
        pd.DataFrame({"feature": feature_names, "importance": importances})
        .sort_values("importance", ascending=False)
        .head(top_n)
        .sort_values("importance")
    )

    plt.figure(figsize=(9, 6))
    plt.barh(imp_df["feature"], imp_df["importance"])
    plt.title(f"Importancia de variables — {phase_name}")
    plt.xlabel("importancia")
    plt.tight_layout()
    plt.show()

    return imp_df.sort_values("importance", ascending=False)


for phase_name in phase_config.keys():
    display(plot_rf_importance(all_models, phase_name, top_n=15))

## 9. ETA dinámico por suma de fases

Una vez entrenados los modelos por separado, se puede estimar el ETA como:

```text
ETA_predicho = predicción_fase_1 + predicción_fase_2 + predicción_fase_3
```

Hay dos maneras de plantearlo:

1. **ETA inicial**: predicción completa al activarse el pedido.  
   No se pueden usar duraciones reales de fase 1 ni fase 2.

2. **ETA dinámico**: se actualiza durante el pedido.  
   Al empezar la fase 2 ya conocemos lo ocurrido en fase 1.  
   Al empezar la fase 3 ya conocemos lo ocurrido en fase 2.

La siguiente simulación usa el enfoque dinámico, coherente con el modelaje por fases.

In [ ]:
def common_clean_data(data):
    """Dataset común donde las tres fases son válidas."""
    common = data.dropna(subset=[
        "phase_1_assignment_minutes",
        "phase_2_pickup_minutes",
        "phase_3_last_mile_minutes",
        "activation_time_local"
    ]).copy()

    for target in [
        "phase_1_assignment_minutes",
        "phase_2_pickup_minutes",
        "phase_3_last_mile_minutes"
    ]:
        common = common[common[target] >= 0]
        common = common[common[target] <= common[target].quantile(0.995)]

    common["eta_sum_real"] = (
        common["phase_1_assignment_minutes"]
        + common["phase_2_pickup_minutes"]
        + common["phase_3_last_mile_minutes"]
    )

    return common


def fit_predict_single_model(train_df, test_df, config, model):
    target = config["target"]
    features = config["numeric"] + config["categorical"]

    pipe = Pipeline(steps=[
        ("preprocess", make_preprocessor(config["numeric"], config["categorical"])),
        ("model", model)
    ])

    pipe.fit(train_df[features], train_df[target])
    pred = pipe.predict(test_df[features])

    return pred, pipe


common_df = common_clean_data(df)
train_common, test_common = temporal_split(common_df)

eta_predictions = pd.DataFrame(index=test_common.index)
fitted_dynamic_models = {}

for phase_name, config in phase_config.items():
    pred, pipe = fit_predict_single_model(
        train_common,
        test_common,
        config,
        RandomForestRegressor(
            n_estimators=200,
            min_samples_leaf=20,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE
        )
    )

    eta_predictions[phase_name] = pred
    fitted_dynamic_models[phase_name] = pipe

eta_predictions["eta_pred_dynamic"] = eta_predictions.sum(axis=1)
eta_predictions["eta_real_sum"] = test_common["eta_sum_real"].values
eta_predictions["eta_real_total"] = test_common["delivery_time_minutes"].values

dynamic_eta_metrics = regression_metrics(
    eta_predictions["eta_real_sum"],
    eta_predictions["eta_pred_dynamic"]
)

dynamic_eta_metrics

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(
    eta_predictions["eta_real_sum"],
    eta_predictions["eta_pred_dynamic"],
    alpha=0.2,
    s=10
)

max_value = max(
    eta_predictions["eta_real_sum"].max(),
    eta_predictions["eta_pred_dynamic"].max()
)

plt.plot([0, max_value], [0, max_value], linestyle="--")
plt.xlabel("ETA real por suma de fases")
plt.ylabel("ETA predicho por suma de modelos")
plt.title("ETA dinámico: real vs predicho")
plt.tight_layout()
plt.show()

## 10. Modelo alternativo para fase 1: problema de cero-inflación

La fase 1 tiene muchos casos con valor 0 tras el clipping, porque el courier ya había empezado o estaba preasignado.

En un trabajo más avanzado, esta fase podría tratarse como un problema en dos pasos:

1. Clasificación: ¿habrá espera positiva?
2. Regresión: si hay espera, ¿cuántos minutos?

No se implementa como modelo principal para mantener el pipeline comparable, pero es una manipulación razonable para justificar en la memoria.

In [ ]:
phase1_zero_share = (df["phase_1_assignment_minutes"] == 0).mean()
print(f"Porcentaje de fase 1 igual a cero: {phase1_zero_share:.2%}")

df[["phase_1_raw_minutes", "phase_1_assignment_minutes", "phase_1_preassigned_flag"]].describe()

## 11. Conclusiones para defender el trabajo

El modelaje por fases es más sólido que predecir directamente el ETA total porque separa mecanismos operativos distintos:

### Fase 1 — Activación a inicio del courier
Representa asignación, disponibilidad y saturación.  
Tiene sentido que dependan más la hora, saturación y zona que la distancia final.

### Fase 2 — Inicio del courier a pickup
Mezcla desplazamiento al comercio, preparación, espera y eficiencia de tienda.  
Tiene sentido incluir vertical, tienda, número de productos y tipo de transporte.

### Fase 3 — Pickup a entrega
Es la última milla pura.  
La distancia, transporte, saturación y horario deberían ser las variables más relevantes.

### Ventaja académica
Aunque el score global no mejore de forma extrema, la estructura es más explicable, reduce leakage y permite defender mejor las transformaciones.

### Manipulaciones adicionales recomendables
- Clusterizar zonas geográficas con KMeans.
- Añadir clima si estuviera disponible.
- Crear medias históricas por tienda/courier usando únicamente pedidos anteriores.
- Hacer target encoding temporal, no global, para tiendas con alta cardinalidad.
- Modelar fase 1 con clasificación + regresión por la alta proporción de ceros.

## 12. Checklist anti-leakage

- No se usan `delivery_time_local`, `pickup_time_local` ni `courier_started_order_local` como features directas.
- Solo se usan duraciones previas cuando ya serían conocidas en esa fase.
- No se usa `rating`.
- No se usa `bad_rating_reason`.
- El split de train/test es temporal.
- Las categorías raras se agrupan para evitar memorizar tiendas específicas.